<a href="https://colab.research.google.com/github/SabrinaZ600/AI-four-dimension-project/blob/main/04AI_Response_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# Import Libraries
# ==========================================================

import os
import re
import json
import numpy as np
import pandas as pd

from IPython.display import display

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================================
# Project Directory
# ==========================================================

PROJECT_DIR = "/content/drive/MyDrive/AI_Brand_Project"

DATA_DIR = os.path.join(PROJECT_DIR, "data")

OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")

LOG_DIR = os.path.join(PROJECT_DIR, "logs")

In [ ]:
RAW_RESPONSE_FILE = os.path.join(
    DATA_DIR,
    "responses_raw.csv"
)

responses_df = pd.read_csv(
    RAW_RESPONSE_FILE
)

print("Responses Loaded:", len(responses_df))

Responses Loaded: 14


In [ ]:
responses_df.head()

,Run_ID,Model,Prompt_ID,Prompt_Type,Category,Prompt,Response,JSON_File,Response_Time,Timestamp,Status,Error_Message
0,20260731_065706,llama-3.3-70b-versatile,1,Open,General,What are the best wireless over-ear noise-canc...,"After reviewing and comparing various models, ...",prompt_001.json,1.69,2026-07-31 06:58:04.110701,Success,NaN
1,20260731_065706,llama-3.3-70b-versatile,2,Open,General,Which wireless headphones would you recommend ...,"For most people, I highly recommend the Sony W...",prompt_002.json,1.90,2026-07-31 06:58:11.051780,Success,NaN
2,20260731_065706,llama-3.3-70b-versatile,3,Open,General,What are the top five wireless headphones on t...,"As a consumer electronics reviewer, I've had t...",prompt_003.json,1.75,2026-07-31 06:58:17.826474,Success,NaN
3,20260731_065706,llama-3.3-70b-versatile,4,Open,Value,Which wireless headphones offer the best value...,When it comes to wireless headphones with acti...,prompt_004.json,2.11,2026-07-31 06:58:24.980226,Success,NaN
4,20260731_065706,llama-3.3-70b-versatile,5,Open,ANC,Which wireless headphones have the best active...,After reviewing and testing numerous wireless ...,prompt_005.json,1.49,2026-07-31 06:58:31.495457,Success,NaN


In [ ]:
responses_df = responses_df[
    responses_df["Status"]=="Success"
].copy()

responses_df.reset_index(
    drop=True,
    inplace=True
)

print("Valid responses:",len(responses_df))

Valid responses: 14


In [ ]:
BRANDS = [

    "Sony",

    "Bose",

    "Sennheiser",

    "SoundCore",

    "JBL",

    "Nothing",

    "Edifier"

]

In [ ]:
ALIASES = {

    "Sony":"Sony",

    "Bose":"Bose",

    "JBL":"JBL",

    "Nothing":"Nothing",

    "Edifier":"Edifier",

    "Sennheiser":"Sennheiser",

    "Soundcore":"SoundCore",

    "SoundCore":"SoundCore",

    "Anker Soundcore":"SoundCore",

    "Soundcore by Anker":"SoundCore"

}

In [ ]:
print(
    responses_df.loc[0,"Response"]
)

After reviewing and comparing various models, I highly recommend the following wireless over-ear noise-cancelling headphones:

1. **Sony WH-1000XM5**: These headphones are the latest iteration of Sony's flagship noise-cancelling series. They offer exceptional sound quality, industry-leading noise cancellation, and a sleek design. The WH-1000XM5 features advanced noise-sensing technology, which adapts to your environment to provide optimal noise cancellation. They also have a long battery life of up to 30 hours and quick charging capabilities.

2. **Bose QuietComfort 45**: Bose is a well-known brand in the noise-cancelling headphone market, and the QuietComfort 45 is one of their best offerings. These headphones provide excellent sound quality, comfortable fit, and advanced noise-rejection technology. They also have a long battery life of up to 24 hours and a sleek, foldable design.

3. **Sennheiser HD 4.50 BT**: Sennheiser is a renowned audio brand, and the HD 4.50 BT is a great option

Setion 2:

In [ ]:
# ==========================================================
# Copy Response
# ==========================================================

responses_df["Clean_Response"] = (
    responses_df["Response"]
    .fillna("")
    .astype(str)
)

In [ ]:
responses_df["Clean_Response"] = (
    responses_df["Clean_Response"]

    .str.replace("\r"," ",regex=False)

    .str.replace("\n\n","\n",regex=False)

    .str.replace("\t"," ",regex=False)
)

In [ ]:
responses_df["Clean_Response"] = (
    responses_df["Clean_Response"]

    .str.replace(r"\*\*","",regex=True)

    .str.replace(r"#+","",regex=True)

    .str.replace("•","-",regex=False)
)

In [ ]:
responses_df["Clean_Response"] = (

    responses_df["Clean_Response"]

    .str.replace(r"[ ]+"," ",regex=True)

    .str.strip()

)

In [ ]:
print(responses_df.loc[0,"Clean_Response"])

After reviewing and comparing various models, I highly recommend the following wireless over-ear noise-cancelling headphones:
1. Sony WH-1000XM5: These headphones are the latest iteration of Sony's flagship noise-cancelling series. They offer exceptional sound quality, industry-leading noise cancellation, and a sleek design. The WH-1000XM5 features advanced noise-sensing technology, which adapts to your environment to provide optimal noise cancellation. They also have a long battery life of up to 30 hours and quick charging capabilities.
2. Bose QuietComfort 45: Bose is a well-known brand in the noise-cancelling headphone market, and the QuietComfort 45 is one of their best offerings. These headphones provide excellent sound quality, comfortable fit, and advanced noise-rejection technology. They also have a long battery life of up to 24 hours and a sleek, foldable design.
3. Sennheiser HD 4.50 BT: Sennheiser is a renowned audio brand, and the HD 4.50 BT is a great option for those look

In [ ]:
# Section 3
# ==========================================================
# Brand Dictionary
# ==========================================================

PRODUCT_ALIASES = {

    "Sony":[
        "Sony",
        "WH-1000XM6",
        "WH-1000XM5",
        "WH1000XM5",
        "WH-1000XM4",
        "WH1000XM4",
        "XM6",
        "XM5",
        "XM4"
    ],

    "Bose":[
        "Bose",
        "QuietComfort Ultra",
        "QuietComfort",
        "QC Ultra",
        "QC45",
        "QuietComfort 45"
    ],

    "Sennheiser":[
        "Sennheiser",
        "Momentum 4",
        "Momentum 4 Wireless",
        "Momentum Wireless"
    ],

    "SoundCore":[
        "SoundCore",
        "Soundcore",
        "Anker Soundcore",
        "Space One Pro",
        "Space One",
        "Q45",
        "Life Q35"
    ],

    "JBL":[
        "JBL",
        "Tour One M3",
        "Tour One M2",
        "Live 770NC",
        "Live 660NC"
    ],

    "Nothing":[
        "Nothing",
        "Headphone (1)",
        "Nothing Headphone"
    ],

    "Edifier":[
        "Edifier",
        "WH950NB",
        "W860NB",
        "W820NB"
    ]

}

In [ ]:
# ==========================================================
# Reverse Lookup
# ==========================================================

alias_to_brand = {}

for brand, aliases in PRODUCT_ALIASES.items():

    for alias in aliases:

        alias_to_brand[alias.lower()] = brand

print("Aliases:", len(alias_to_brand))

Aliases: 37


In [ ]:
# ==========================================================
# Extract Brands
# ==========================================================

import re

def extract_brands(text):

    text_lower = text.lower()

    found = []

    for alias, brand in alias_to_brand.items():

        if re.search(r"\b" + re.escape(alias) + r"\b", text_lower):

            found.append(brand)

    found = list(dict.fromkeys(found))

    return found

In [ ]:
brands = extract_brands(

responses_df.loc[0,"Clean_Response"]

)

brands

['Sony', 'Bose', 'Sennheiser']

In [ ]:
responses_df["Brands"] = (

responses_df["Clean_Response"]

.apply(extract_brands)

)

responses_df[

["Prompt_ID","Brands"]

].head()

,Prompt_ID,Brands
0,1,"[Sony, Bose, Sennheiser]"
1,2,"[Sony, Bose, Sennheiser, SoundCore]"
2,3,"[Sony, Bose, Sennheiser, SoundCore]"
3,4,"[Sony, Bose, Sennheiser, SoundCore]"
4,5,"[Sony, Bose, Sennheiser]"


In [ ]:
records = []

for _, row in responses_df.iterrows():

    for brand in PRODUCT_ALIASES.keys():

        records.append({

            "Prompt_ID": row["Prompt_ID"],

            "Brand": brand,

            "Mention": int(

                brand in row["Brands"]

            )

        })

mention_df = pd.DataFrame(records)

mention_df.head()

,Prompt_ID,Brand,Mention
0,1,Sony,1
1,1,Bose,1
2,1,Sennheiser,1
3,1,SoundCore,0
4,1,JBL,0


In [ ]:
# Section 4:
# ==========================================================
# Extract Brand Rankings
# ==========================================================

import re

def extract_rankings(text):

    rankings = {}

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # 匹配 "1. xxx"、"2) xxx"、"3 - xxx"
        match = re.match(r"^(\d+)[\.\)\-:]\s*(.*)", line)

        if not match:
            continue

        rank = int(match.group(1))
        content = match.group(2).lower()

        for alias, brand in alias_to_brand.items():

            if re.search(r"\b" + re.escape(alias) + r"\b", content):

                if brand not in rankings:
                    rankings[brand] = rank

    return rankings

In [ ]:
ranking = extract_rankings(
    responses_df.loc[0, "Clean_Response"]
)

ranking

{'Sony': 1, 'Bose': 2, 'Sennheiser': 3}

In [ ]:
responses_df["Rankings"] = (
    responses_df["Clean_Response"]
    .apply(extract_rankings)
)

responses_df[
    ["Prompt_ID", "Rankings"]
].head()

,Prompt_ID,Rankings
0,1,"{'Sony': 1, 'Bose': 2, 'Sennheiser': 3}"
1,2,{'Sony': 1}
2,3,"{'Sony': 1, 'Bose': 2, 'Sennheiser': 3, 'Sound..."
3,4,"{'Sony': 1, 'Bose': 2, 'Sennheiser': 3, 'Sound..."
4,5,"{'Bose': 1, 'Sony': 2, 'Sennheiser': 3}"


In [ ]:
ranking_records = []

for _, row in responses_df.iterrows():

    ranking_dict = row["Rankings"]

    for brand in PRODUCT_ALIASES.keys():

        ranking_records.append({

            "Prompt_ID": row["Prompt_ID"],

            "Brand": brand,

            "Mention": int(brand in row["Brands"]),

            "Rank": ranking_dict.get(brand, np.nan)

        })

ranking_df = pd.DataFrame(ranking_records)

ranking_df.head(15)

,Prompt_ID,Brand,Mention,Rank
0,1,Sony,1,1.0
1,1,Bose,1,2.0
2,1,Sennheiser,1,3.0
3,1,SoundCore,0,NaN
4,1,JBL,0,NaN
5,1,Nothing,0,NaN
6,1,Edifier,0,NaN
7,2,Sony,1,1.0
8,2,Bose,1,NaN
9,2,Sennheiser,1,NaN


In [ ]:
ranking_df.groupby("Brand").agg(

    Mentions=("Mention", "sum"),

    Avg_Rank=("Rank", "mean")

).sort_values(

    "Mentions",

    ascending=False

)

,Mentions,Avg_Rank
Brand,,
Bose,14,2.000000
Sony,14,1.538462
Sennheiser,13,2.700000
SoundCore,11,4.000000
Edifier,6,5.200000
JBL,6,4.200000
Nothing,6,6.400000


In [ ]:
RANKING_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "brand_rankings.csv"
)

ranking_df.to_csv(
    RANKING_OUTPUT,
    index=False
)

print("Saved to:")
print(RANKING_OUTPUT)

Saved to:
/content/drive/MyDrive/AI_Brand_Project/output/brand_rankings.csv


In [ ]:
ranking_df.head(20)

,Prompt_ID,Brand,Mention,Rank
0,1,Sony,1,1.0
1,1,Bose,1,2.0
2,1,Sennheiser,1,3.0
3,1,SoundCore,0,NaN
4,1,JBL,0,NaN
5,1,Nothing,0,NaN
6,1,Edifier,0,NaN
7,2,Sony,1,1.0
8,2,Bose,1,NaN
9,2,Sennheiser,1,NaN
